In [5]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import random
import os


# Task 1: Denoising Autoencoder

This notebook implements a denoising autoencoder using grayscale natural images (64×64).

**Objectives:**
- Train a denoising autoencoder on natural images
- Corrupt input with Gaussian noise
- Evaluate reconstruction quality on unseen noisy data
- Visualize training/test performance and reconstructed images

## 1. Load and Preprocess Dataset (Grayscale Natural Images)
- We'll use CIFAR10 (houses, animals, etc.) and convert to grayscale and resize to 64x64

In [7]:
# Data preprocessing
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Download CIFAR10 and use first 100 grayscale images
dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
subset = torch.utils.data.Subset(dataset, range(100))
train_size = 80
train_set, test_set = random_split(subset, [train_size, 20])

batch_size = 10
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

## 2. Add Gaussian Noise to Input Images
- Noise level is visually noticeable but doesn't destroy image content

In [9]:
def add_gaussian_noise(images, mean=0.0, std=0.2):
    noise = torch.randn_like(images) * std + mean
    noisy_images = images + noise
    return torch.clamp(noisy_images, 0., 1.)

## 3. Define the Autoencoder Model

In [11]:
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

model = Autoencoder()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## 4. Train the Autoencoder

In [13]:
train_losses = []
test_losses = []

for epoch in range(10):
    model.train()
    running_loss = 0.0
    for images, _ in train_loader:
        noisy = add_gaussian_noise(images)
        outputs = model(noisy)
        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    train_losses.append(running_loss / len(train_loader))

    # Test loss
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for images, _ in test_loader:
            noisy = add_gaussian_noise(images)
            outputs = model(noisy)
            test_loss += criterion(outputs, images).item()
    test_losses.append(test_loss / len(test_loader))

## 5. Visualize Loss Curves

In [ ]:
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training vs Test Loss')
plt.legend()
plt.grid(True)
plt.show()

## 6. Visualize Reconstructed Images from Test Set

In [2]:
model.eval()
images, _ = next(iter(test_loader))
noisy = add_gaussian_noise(images)
with torch.no_grad():
    outputs = model(noisy)

# Show original, noisy, and reconstructed
def show_images(original, noisy, reconstructed):
    fig, axes = plt.subplots(3, 10, figsize=(15, 5))
    for i in range(10):
        axes[0][i].imshow(original[i][0], cmap='gray')
        axes[1][i].imshow(noisy[i][0], cmap='gray')
        axes[2][i].imshow(reconstructed[i][0], cmap='gray')
        for ax in axes[:, i]: ax.axis('off')
    axes[0][0].set_ylabel('Original')
    axes[1][0].set_ylabel('Noisy')
    axes[2][0].set_ylabel('Denoised')
    plt.tight_layout()
    plt.show()

show_images(images, noisy, outputs)

NameError: name 'model' is not defined